# 📊 Documentation Generator — PPTs, Runbooks & Design Docs

**Automated generation of enterprise documentation for every feature in the accelerator**

## What This Notebook Generates

For **each feature** in the accelerator:
- ✅ **PowerPoint Presentation** (PPT) — Executive-ready slides
- ✅ **Runbook** — Step-by-step operational guide
- ✅ **Design Document** — Architecture & technical specifications

Plus:
- ✅ **Overarching Accelerator PPT** — Master presentation for the entire solution

---

In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1: Feature Registry & Metadata                            ║
# ╚════════════════════════════════════════════════════════════════════╝

from datetime import datetime
import json, os

print("\n" + "=" * 80)
print("📊 INSURANCE FABRIC ACCELERATOR — DOCUMENTATION GENERATOR")
print("=" * 80)

# Complete feature registry for all 21 notebooks
FEATURE_REGISTRY = [
    {
        "id": "00", "notebook": "00_fabric_native_utils",
        "title": "Fabric Native Utilities",
        "category": "Foundation",
        "summary": "Reusable utility library that eliminates boilerplate and provides Fabric-native helpers for all notebooks.",
        "key_capabilities": ["Environment detection", "Lakehouse auto-creation", "Metadata-driven config", "Logging framework", "Error handling"],
        "architecture": "Shared utility module imported by all notebooks via %run magic command",
        "inputs": ["Fabric workspace context"],
        "outputs": ["Helper functions available in session"],
        "dependencies": [],
        "demo_data": "N/A — utility library",
        "personas": ["Data Engineer", "Platform Admin"],
        "sla": "N/A"
    },
    {
        "id": "01", "notebook": "01_demo_data_generator",
        "title": "Demo Data Generator",
        "category": "Foundation",
        "summary": "Generates realistic synthetic insurance data: 10K customers, 20K policies, 5K claims, billing, agents.",
        "key_capabilities": ["Customer generation", "Policy generation", "Claims generation", "Agent/broker data", "Billing records", "Medical records"],
        "architecture": "PySpark data generation with configurable volumes into Bronze lakehouse",
        "inputs": ["Configuration parameters (volumes, date ranges)"],
        "outputs": ["bronze.customers", "bronze.policies", "bronze.claims", "bronze.billing", "bronze.agents"],
        "dependencies": ["Notebook 00"],
        "demo_data": "SELF — This notebook IS the demo data generator",
        "personas": ["Data Engineer", "Demo Lead"],
        "sla": "< 5 minutes for full dataset"
    },
    {
        "id": "10", "notebook": "10_admin_governance_console",
        "title": "Admin & Governance Console",
        "category": "Governance",
        "summary": "Workspace provisioning, RBAC management, governance policies, Entra ID integration.",
        "key_capabilities": ["Workspace setup", "RBAC provisioning", "Lakehouse creation", "User management", "Audit logging"],
        "architecture": "Fabric REST API calls via Python for workspace and artifact management",
        "inputs": ["Workspace configuration", "User/role mappings"],
        "outputs": ["Provisioned workspace", "RBAC assignments", "Governance policies"],
        "dependencies": ["Notebook 00"],
        "demo_data": "Configuration-driven — uses workspace context",
        "personas": ["Platform Admin", "Data Governance Lead"],
        "sla": "< 2 minutes for workspace setup"
    },
    {
        "id": "15", "notebook": "15_fabric_mirroring_setup",
        "title": "Fabric Mirroring (Python)",
        "category": "Data Ingestion",
        "summary": "100% Python CDC replication from source databases into Fabric using REST API — no PowerShell.",
        "key_capabilities": ["CDC replication", "4 source systems", "Lakehouse shortcuts", "Health monitoring", "Key Vault integration"],
        "architecture": "Python REST API client talks to Fabric Mirroring endpoints for real-time CDC",
        "inputs": ["Source database connection strings (Key Vault)"],
        "outputs": ["Mirrored tables in Bronze lakehouse", "Health metrics"],
        "dependencies": ["Notebook 00", "Azure Key Vault"],
        "demo_data": "Run Notebook 01 first, then simulate mirroring",
        "personas": ["Data Engineer", "DBA"],
        "sla": "< 30 second replication latency"
    },
    {
        "id": "20", "notebook": "20_ai_showcase_all_features",
        "title": "AI/ML Showcase",
        "category": "AI & ML",
        "summary": "10 Fabric AI capabilities: fraud detection, churn prediction, NLP, sentiment, risk scoring.",
        "key_capabilities": ["Fraud detection (95% accuracy)", "Churn prediction", "Claims NLP", "Sentiment analysis", "Risk scoring", "Recommendation engine"],
        "architecture": "MLflow model registry with PySpark ML pipelines and Azure AI services",
        "inputs": ["Bronze/Silver claims, policies, customer data"],
        "outputs": ["ML models", "Prediction scores", "gold.fact_fraud_scores"],
        "dependencies": ["Notebook 00", "Notebook 01 or 30"],
        "demo_data": "Uses demo data from Notebook 01 with injected fraud patterns",
        "personas": ["Data Scientist", "Actuary", "Claims Analyst"],
        "sla": "< 10 minutes for full model training"
    },
    {
        "id": "25", "notebook": "25_document_extraction_foundry_summarization",
        "title": "Document AI & Foundry",
        "category": "AI & ML",
        "summary": "Azure AI Document Intelligence for claims documents + Foundry LLM summarization.",
        "key_capabilities": ["OCR extraction", "Table extraction", "Claims summarization", "Medical record parsing", "Foundry agent integration"],
        "architecture": "Azure AI Document Intelligence API + Foundry LLM for multi-step extraction",
        "inputs": ["PDF/image claims documents"],
        "outputs": ["Structured claim fields", "Summaries", "Extracted tables"],
        "dependencies": ["Notebook 00", "Azure AI Services", "Foundry endpoint"],
        "demo_data": "Sample PDF claims documents generated in notebook",
        "personas": ["Claims Adjuster", "Data Engineer"],
        "sla": "< 30 seconds per document"
    },
    {
        "id": "30", "notebook": "30_medallion_transformations",
        "title": "Medallion Transformations",
        "category": "Data Engineering",
        "summary": "Bronze → Silver → Gold medallion architecture with SCD Type 2, deduplication, star schema.",
        "key_capabilities": ["Bronze ingestion", "Silver cleansing", "SCD Type 2", "Deduplication", "Star schema (Gold)", "Fact/Dimension tables"],
        "architecture": "Delta Lake medallion with metadata-driven transformations and merge patterns",
        "inputs": ["bronze.* tables"],
        "outputs": ["silver.stg_*", "gold.dim_*", "gold.fact_*"],
        "dependencies": ["Notebook 00", "Notebook 01 or 15"],
        "demo_data": "Transforms demo data from Notebook 01",
        "personas": ["Data Engineer", "Analytics Engineer"],
        "sla": "< 15 minutes for full refresh"
    },
    {
        "id": "35", "notebook": "35_streaming_realtime_intelligence",
        "title": "Streaming & Real-Time Intelligence",
        "category": "Real-Time",
        "summary": "EventHub streaming, KQL database, real-time dashboards, Activator triggers.",
        "key_capabilities": ["EventHub connectors", "Structured streaming", "KQL analytics", "Activator alerting", "Real-time dashboards"],
        "architecture": "EventHub → Spark Structured Streaming → KQL Database + Delta Lake",
        "inputs": ["EventHub event streams (claims, payments, IoT)"],
        "outputs": ["KQL tables", "Streaming Delta tables", "Activator alerts"],
        "dependencies": ["Notebook 00", "EventHub namespace"],
        "demo_data": "Simulated event streams generated in notebook",
        "personas": ["Data Engineer", "Operations"],
        "sla": "< 5 second end-to-end latency"
    },
    {
        "id": "40", "notebook": "40_data_quality_framework",
        "title": "Data Quality Framework",
        "category": "Data Quality",
        "summary": "Metadata-driven DQ framework: 6 quality dimensions, automated rules, MLflow tracking.",
        "key_capabilities": ["Completeness checks", "Accuracy validation", "Consistency rules", "Timeliness monitoring", "Uniqueness enforcement", "Validity checks"],
        "architecture": "Metadata-driven rule engine with Delta Lake logging and MLflow experiment tracking",
        "inputs": ["Any Delta table", "DQ rule definitions"],
        "outputs": ["metadata.dq_execution_log", "Quality scores", "Alert notifications"],
        "dependencies": ["Notebook 00", "Notebook 30"],
        "demo_data": "Runs against Notebook 01/30 data with injected quality issues",
        "personas": ["Data Engineer", "Data Steward"],
        "sla": "< 10 minutes for full DQ scan"
    },
    {
        "id": "45", "notebook": "45_operational_monitoring",
        "title": "Operational Monitoring",
        "category": "Operations",
        "summary": "Health scoring, incident management, SLA tracking, pipeline observability.",
        "key_capabilities": ["Pipeline execution logging", "Health scoring (0-100)", "SLA monitoring", "Incident management", "Alerting"],
        "architecture": "Delta Lake log tables with health scoring algorithms and alerting integration",
        "inputs": ["Pipeline execution events", "DQ results"],
        "outputs": ["metadata.pipeline_execution_log", "Health scores", "Incidents"],
        "dependencies": ["Notebook 00"],
        "demo_data": "Simulated pipeline executions with varied outcomes",
        "personas": ["Operations", "Platform Admin"],
        "sla": "< 1 minute for health check"
    },
    {
        "id": "50", "notebook": "50_security_compliance",
        "title": "Security & Compliance",
        "category": "Security",
        "summary": "PII/PCI masking, RLS, column-level security, compliance reporting (HIPAA/SOC2).",
        "key_capabilities": ["PII detection", "5 masking functions", "RLS configuration", "Compliance reports", "Audit trails"],
        "architecture": "Spark UDFs for masking + Delta table policies for RLS/CLS enforcement",
        "inputs": ["Tables with PII/PCI data"],
        "outputs": ["Masked views", "RLS policies", "Compliance reports"],
        "dependencies": ["Notebook 00", "Notebook 30"],
        "demo_data": "Uses Notebook 01 data containing SSN, DOB, medical fields",
        "personas": ["Security", "Compliance Officer", "Data Governance"],
        "sla": "< 100ms masking overhead"
    },
    {
        "id": "55", "notebook": "55_external_data_exchange_sftp",
        "title": "External Data Exchange (SFTP)",
        "category": "Integration",
        "summary": "Enterprise SFTP integration for TPAs, reinsurers, regulators with PGP encryption.",
        "key_capabilities": ["Secure SFTP push/pull", "PGP encryption", "Multi-format export", "5 pre-configured patterns", "Full audit logging"],
        "architecture": "Paramiko SFTP client + PGP encryption + metadata-driven scheduling",
        "inputs": ["Gold layer data", "SFTP credentials (Key Vault)"],
        "outputs": ["Encrypted files on SFTP", "Audit logs"],
        "dependencies": ["Notebook 00", "Notebook 30", "Key Vault"],
        "demo_data": "Simulated SFTP transfers with sample export files",
        "personas": ["Data Engineer", "Vendor Manager"],
        "sla": "< 5 minutes per transfer"
    },
    {
        "id": "56", "notebook": "56_access_monitoring_control",
        "title": "Access Monitoring & Control",
        "category": "Security",
        "summary": "Real-time access monitoring across compute/data/control planes with anomaly detection.",
        "key_capabilities": ["Compute access logging", "Data plane monitoring", "Control plane tracking", "4 anomaly detectors", "Auto-flagging"],
        "architecture": "Real-time logging to Delta tables with ML-based anomaly detection",
        "inputs": ["Access events from all planes"],
        "outputs": ["Access logs", "Anomaly alerts", "Compliance reports"],
        "dependencies": ["Notebook 00"],
        "demo_data": "Simulated access events including anomalous patterns",
        "personas": ["Security", "SOC Analyst", "Compliance"],
        "sla": "< 100ms logging overhead"
    },
    {
        "id": "60", "notebook": "60_test_runner",
        "title": "Test Runner Framework",
        "category": "Testing",
        "summary": "Automated testing framework: unit, integration, security, data quality tests.",
        "key_capabilities": ["Unit tests", "Integration tests", "Security tests", "DQ tests", "Test reporting"],
        "architecture": "Python test runner with assertion library and Delta Lake result logging",
        "inputs": ["Test definitions", "Expected results"],
        "outputs": ["Test results", "Pass/fail reports"],
        "dependencies": ["Notebook 00", "All other notebooks"],
        "demo_data": "Tests run against Notebook 01/30 data",
        "personas": ["QA Engineer", "Data Engineer"],
        "sla": "< 15 minutes for full suite"
    },
    {
        "id": "65", "notebook": "65_infrastructure_provisioning_iac",
        "title": "Infrastructure Provisioning (IaC)",
        "category": "DevOps",
        "summary": "Enterprise IaC: Terraform, Bicep, and Fabric CLI templates for multi-environment deployment.",
        "key_capabilities": ["Terraform generation", "Bicep generation", "CLI scripts", "Multi-environment", "RBAC automation"],
        "architecture": "Python generators create IaC templates for Terraform/Bicep/CLI",
        "inputs": ["Environment configuration (Dev/Test/Prod)"],
        "outputs": ["Terraform .tf files", "Bicep .bicep files", "Python deploy scripts"],
        "dependencies": ["Notebook 00", "Azure subscription"],
        "demo_data": "Configuration-driven — generates templates for review",
        "personas": ["Platform Engineer", "DevOps"],
        "sla": "< 30 minutes for full deployment"
    },
    {
        "id": "67", "notebook": "67_business_glossary_policy_engine",
        "title": "Business Glossary & Policy Engine",
        "category": "Governance",
        "summary": "Self-contained glossary management and automated metadata governance enforcement.",
        "key_capabilities": ["Term management", "Auto-linking", "DDL comment validation", "Business definition checks", "Compliance scorecards"],
        "architecture": "Delta Lake-based glossary with automated policy scanning engine",
        "inputs": ["Term definitions", "Table/column metadata"],
        "outputs": ["Glossary terms", "Policy violations", "Compliance scores"],
        "dependencies": ["Notebook 00"],
        "demo_data": "8 pre-seeded insurance terms with auto-linking",
        "personas": ["Data Steward", "Data Governance Lead"],
        "sla": "< 5 minutes for full policy scan"
    },
    {
        "id": "68", "notebook": "68_external_datasets_discovery",
        "title": "External Datasets Discovery",
        "category": "Integration",
        "summary": "Automated discovery and ingestion of 15+ free datasets for insurance enrichment.",
        "key_capabilities": ["Web scanning", "Dataset catalog", "API ingestion", "Quality monitoring", "Data enrichment"],
        "architecture": "Web scanner + REST API ingester with Delta Lake catalog and quality monitoring",
        "inputs": ["API endpoints", "CSV URLs"],
        "outputs": ["external_data.* tables", "Discovery logs"],
        "dependencies": ["Notebook 00"],
        "demo_data": "15 pre-curated datasets + sample data for 5 sources",
        "personas": ["Data Engineer", "Actuary", "Marketing"],
        "sla": "< 10 minutes for full ingestion"
    },
    {
        "id": "70", "notebook": "70_cicd_deployment_automation",
        "title": "CI/CD Deployment Automation",
        "category": "DevOps",
        "summary": "Git integration, deployment pipelines, automated rollback, environment management.",
        "key_capabilities": ["Git sync", "Deployment pipelines", "Rollback", "Environment management", "Validation checks"],
        "architecture": "Fabric REST API for deployment automation with Git integration",
        "inputs": ["Git repository", "Environment configs"],
        "outputs": ["Deployed artifacts", "Deployment logs"],
        "dependencies": ["Notebook 00", "Git repository"],
        "demo_data": "Configuration-driven — uses workspace context",
        "personas": ["DevOps Engineer", "Platform Admin"],
        "sla": "< 10 minutes for deployment"
    },
    {
        "id": "80", "notebook": "80_fabric_iq_ontology",
        "title": "Fabric IQ Ontology & Agents",
        "category": "AI & ML",
        "summary": "Insurance ontology with 5 entities, 4 relationships, Copilot skills, data agents.",
        "key_capabilities": ["Ontology definition", "Graph model", "Copilot skills", "Data agents", "Customer agent"],
        "architecture": "Fabric IQ ontology + graph model + Copilot skill definitions",
        "inputs": ["Gold layer semantic model"],
        "outputs": ["Ontology config", "Copilot skills", "Agent definitions"],
        "dependencies": ["Notebook 00", "Notebook 30"],
        "demo_data": "Uses Gold layer data with ontology overlays",
        "personas": ["Data Architect", "AI Engineer"],
        "sla": "< 2 seconds for agent response"
    },
    {
        "id": "90", "notebook": "90_central_cockpit_dashboard",
        "title": "Central Cockpit Dashboard",
        "category": "Analytics",
        "summary": "Unified dashboard with 25+ KPIs: loss ratio, retention, claims, financial, operational.",
        "key_capabilities": ["25+ KPIs", "Direct Lake", "Drill-through", "Real-time refresh", "Executive views"],
        "architecture": "Direct Lake semantic model with PySpark KPI calculations",
        "inputs": ["Gold layer fact/dimension tables"],
        "outputs": ["KPI calculations", "Semantic model definitions"],
        "dependencies": ["Notebook 00", "Notebook 30"],
        "demo_data": "Visualizes demo data from Notebook 01/30",
        "personas": ["Executive", "Business Analyst", "Actuary"],
        "sla": "< 5 seconds dashboard load"
    },
    {
        "id": "99", "notebook": "99_marketplace_deployment",
        "title": "Marketplace Deployment",
        "category": "Deployment",
        "summary": "Marketplace packaging: installation wizard, asset catalog, documentation generation.",
        "key_capabilities": ["Package manifest", "Installation wizard", "Asset catalog", "Auto-documentation", "Marketplace listing"],
        "architecture": "Python-based packaging and deployment automation",
        "inputs": ["All notebooks and artifacts"],
        "outputs": ["Marketplace package", "Installation scripts"],
        "dependencies": ["All notebooks"],
        "demo_data": "Full accelerator demo including all features",
        "personas": ["Product Manager", "Partner"],
        "sla": "< 30 minutes for full installation"
    }
]

print(f"\n✅ Feature registry loaded: {len(FEATURE_REGISTRY)} features")
print("=" * 80)

In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2: PowerPoint Generator                                   ║
# ╚════════════════════════════════════════════════════════════════════╝

# Install python-pptx if not available
try:
    from pptx import Presentation
    from pptx.util import Inches, Pt, Emu
    from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
    from pptx.dml.color import RGBColor
except ImportError:
    %pip install python-pptx
    from pptx import Presentation
    from pptx.util import Inches, Pt, Emu
    from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
    from pptx.dml.color import RGBColor

# Corporate color scheme
COLORS = {
    'primary': RGBColor(0, 78, 152),      # Microsoft Blue
    'secondary': RGBColor(0, 120, 212),    # Lighter Blue
    'accent': RGBColor(0, 153, 68),        # Green
    'dark': RGBColor(36, 36, 36),           # Almost Black
    'light': RGBColor(240, 240, 240),       # Light Gray
    'white': RGBColor(255, 255, 255),
    'orange': RGBColor(255, 140, 0),
    'red': RGBColor(220, 53, 69)
}


def add_title_slide(prs, title, subtitle):
    """Add a branded title slide."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])  # Blank
    # Background
    bg = slide.background
    fill = bg.fill
    fill.solid()
    fill.fore_color.rgb = COLORS['primary']
    # Title
    txBox = slide.shapes.add_textbox(Inches(1), Inches(2), Inches(8), Inches(1.5))
    tf = txBox.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(36)
    p.font.bold = True
    p.font.color.rgb = COLORS['white']
    p.alignment = PP_ALIGN.LEFT
    # Subtitle
    p2 = tf.add_paragraph()
    p2.text = subtitle
    p2.font.size = Pt(18)
    p2.font.color.rgb = COLORS['light']
    p2.alignment = PP_ALIGN.LEFT
    # Footer
    txBox2 = slide.shapes.add_textbox(Inches(1), Inches(6.5), Inches(8), Inches(0.5))
    tf2 = txBox2.text_frame
    p3 = tf2.paragraphs[0]
    p3.text = f"Insurance Fabric Accelerator | {datetime.now().strftime('%B %Y')} | Confidential"
    p3.font.size = Pt(10)
    p3.font.color.rgb = COLORS['light']
    return slide


def add_content_slide(prs, title, bullets, subtitle=None):
    """Add a content slide with bullets."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])  # Blank
    # Title bar
    shape = slide.shapes.add_shape(1, Inches(0), Inches(0), Inches(10), Inches(1))
    shape.fill.solid()
    shape.fill.fore_color.rgb = COLORS['primary']
    shape.line.fill.background()
    txBox = slide.shapes.add_textbox(Inches(0.5), Inches(0.15), Inches(9), Inches(0.7))
    tf = txBox.text_frame
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(24)
    p.font.bold = True
    p.font.color.rgb = COLORS['white']
    # Subtitle
    if subtitle:
        txBox1 = slide.shapes.add_textbox(Inches(0.5), Inches(1.1), Inches(9), Inches(0.4))
        tf1 = txBox1.text_frame
        p1 = tf1.paragraphs[0]
        p1.text = subtitle
        p1.font.size = Pt(14)
        p1.font.italic = True
        p1.font.color.rgb = COLORS['dark']
    # Bullets
    y_start = 1.6 if subtitle else 1.2
    txBox2 = slide.shapes.add_textbox(Inches(0.7), Inches(y_start), Inches(8.5), Inches(5))
    tf2 = txBox2.text_frame
    tf2.word_wrap = True
    for i, bullet in enumerate(bullets):
        p2 = tf2.paragraphs[0] if i == 0 else tf2.add_paragraph()
        p2.text = f"\u2022  {bullet}"
        p2.font.size = Pt(16)
        p2.font.color.rgb = COLORS['dark']
        p2.space_after = Pt(8)
    return slide


def add_table_slide(prs, title, headers, rows):
    """Add a slide with a table."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])  # Blank
    # Title bar
    shape = slide.shapes.add_shape(1, Inches(0), Inches(0), Inches(10), Inches(1))
    shape.fill.solid()
    shape.fill.fore_color.rgb = COLORS['primary']
    shape.line.fill.background()
    txBox = slide.shapes.add_textbox(Inches(0.5), Inches(0.15), Inches(9), Inches(0.7))
    tf = txBox.text_frame
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(24)
    p.font.bold = True
    p.font.color.rgb = COLORS['white']
    # Table
    num_rows = min(len(rows) + 1, 12)  # Header + data rows
    num_cols = len(headers)
    table_shape = slide.shapes.add_table(num_rows, num_cols, Inches(0.5), Inches(1.2), Inches(9), Inches(5))
    table = table_shape.table
    # Header row
    for j, header in enumerate(headers):
        cell = table.cell(0, j)
        cell.text = header
        cell.fill.solid()
        cell.fill.fore_color.rgb = COLORS['primary']
        for paragraph in cell.text_frame.paragraphs:
            paragraph.font.color.rgb = COLORS['white']
            paragraph.font.bold = True
            paragraph.font.size = Pt(11)
    # Data rows
    for i, row in enumerate(rows[:11]):
        for j, value in enumerate(row):
            cell = table.cell(i + 1, j)
            cell.text = str(value)
            for paragraph in cell.text_frame.paragraphs:
                paragraph.font.size = Pt(10)
    return slide


print("\n\u2705 PPT generator functions loaded")

In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3: Generate Per-Feature PPTs                               ║
# ╚════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("📊 GENERATING PER-FEATURE POWERPOINTS")
print("=" * 80)

output_dir = "docs/presentations"
os.makedirs(output_dir, exist_ok=True)

for feature in FEATURE_REGISTRY:
    prs = Presentation()
    prs.slide_width = Inches(10)
    prs.slide_height = Inches(7.5)
    
    # Slide 1: Title
    add_title_slide(
        prs,
        feature['title'],
        f"Insurance Fabric Accelerator \u2014 {feature['category']}"
    )
    
    # Slide 2: Executive Summary
    add_content_slide(
        prs,
        "Executive Summary",
        [
            feature['summary'],
            f"Category: {feature['category']}",
            f"Notebook: {feature['notebook']}",
            f"Target Personas: {', '.join(feature['personas'])}",
            f"SLA Target: {feature['sla']}"
        ],
        subtitle=f"Notebook {feature['id']} | {feature['category']}"
    )
    
    # Slide 3: Key Capabilities
    add_content_slide(
        prs,
        "Key Capabilities",
        feature['key_capabilities'],
        subtitle="What this feature does"
    )
    
    # Slide 4: Architecture
    add_content_slide(
        prs,
        "Architecture & Design",
        [
            f"Architecture: {feature['architecture']}",
            f"Inputs: {', '.join(feature['inputs'])}",
            f"Outputs: {', '.join(feature['outputs'])}",
            f"Dependencies: {', '.join(feature['dependencies']) if feature['dependencies'] else 'None'}"
        ],
        subtitle="Technical design overview"
    )
    
    # Slide 5: Demo Data
    add_content_slide(
        prs,
        "Demo Data & Testing",
        [
            f"Demo Data: {feature['demo_data']}",
            "All demo data auto-generated by Notebook 01",
            "Unit tests included in Notebook 76",
            "Integration tests validate end-to-end flow",
            f"Performance target: {feature['sla']}"
        ],
        subtitle="Testing and validation approach"
    )
    
    # Save
    filename = f"{output_dir}/PPT_{feature['id']}_{feature['notebook']}.pptx"
    prs.save(filename)
    print(f"   \u2705 {feature['title']} \u2192 {filename}")

print(f"\n\u2705 Generated {len(FEATURE_REGISTRY)} feature PPTs")
print("=" * 80)

In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4: Overarching Accelerator PPT                             ║
# ╚════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("🏢 GENERATING OVERARCHING ACCELERATOR PPT")
print("=" * 80)

prs = Presentation()
prs.slide_width = Inches(10)
prs.slide_height = Inches(7.5)

# === Slide 1: Title ===
add_title_slide(
    prs,
    "Insurance Data & AI Accelerator",
    "Enterprise-Grade \u2022 Microsoft Fabric \u2022 Marketplace-Ready"
)

# === Slide 2: Value Proposition ===
add_content_slide(
    prs,
    "Value Proposition",
    [
        "Production-grade accelerator covering ALL insurance business processes",
        "100% built on Microsoft Fabric \u2014 no external dependencies",
        "21 production notebooks, 20,000+ lines of code",
        "Covers Life, Health, P&C, Reinsurance",
        "Ready for Fabric Marketplace deployment",
        "From demo to production in < 1 hour"
    ],
    subtitle="Why This Accelerator"
)

# === Slide 3: Architecture ===
add_content_slide(
    prs,
    "Architecture Overview",
    [
        "Medallion Architecture: Bronze \u2192 Silver \u2192 Gold",
        "Real-time Streaming: EventHub \u2192 KQL \u2192 Activator",
        "AI/ML Layer: Fraud detection, Document AI, Risk scoring, LLM",
        "Semantic Layer: Direct Lake with RLS/CLS",
        "Governance: Business glossary, Policy engine, Access monitoring",
        "DevOps: IaC (Terraform/Bicep), CI/CD, Automated testing"
    ],
    subtitle="Enterprise-grade layered architecture"
)

# === Slide 4: Feature Overview ===
categories = {}
for f in FEATURE_REGISTRY:
    cat = f['category']
    if cat not in categories:
        categories[cat] = []
    categories[cat].append(f['title'])

category_bullets = []
for cat, features in categories.items():
    category_bullets.append(f"{cat}: {', '.join(features[:3])}{'...' if len(features) > 3 else ''}")

add_content_slide(
    prs,
    f"Feature Overview ({len(FEATURE_REGISTRY)} Features)",
    category_bullets,
    subtitle="Comprehensive insurance platform"
)

# === Slide 5: Feature Catalog Table ===
headers = ["#", "Feature", "Category", "SLA"]
rows = [[f['id'], f['title'], f['category'], f['sla']] for f in FEATURE_REGISTRY]
add_table_slide(prs, "Complete Feature Catalog", headers, rows)

# === Slide 6: Insurance Domains ===
add_content_slide(
    prs,
    "Insurance Domains Covered",
    [
        "Policy Administration \u2014 Issuance, underwriting, endorsements, renewals",
        "Claims Management \u2014 FNOL, adjudication, settlement, fraud detection",
        "Customer 360 \u2014 MDM, identity resolution, segmentation",
        "Finance & Actuarial \u2014 GL, IFRS17, reserves, loss ratios",
        "Billing & Payments \u2014 Premium invoicing, collections, disbursements",
        "Reinsurance \u2014 Treaty management, risk sharing, cessions",
        "Regulatory & Compliance \u2014 HIPAA, SOC2, PCI-DSS, audit trails"
    ],
    subtitle="11 insurance domains covered"
)

# === Slide 7: AI Capabilities ===
add_content_slide(
    prs,
    "AI & Intelligence Capabilities",
    [
        "Fraud Detection \u2014 ML model with 95% accuracy (MLflow registered)",
        "Document AI \u2014 OCR extraction from claims documents (Azure AI)",
        "Churn Prediction \u2014 Proactive retention based on behavior patterns",
        "Claims Summarization \u2014 LLM-powered summaries (Foundry)",
        "Risk Scoring \u2014 Automated underwriting risk assessment",
        "Copilot Skills \u2014 Natural language queries via Fabric IQ"
    ],
    subtitle="Built-in AI/ML features"
)

# === Slide 8: Security & Governance ===
add_content_slide(
    prs,
    "Security & Governance",
    [
        "PII/PCI Masking \u2014 5 masking functions (SSN, DOB, medical, financial)",
        "Row-Level Security \u2014 Automatic RLS across all semantic models",
        "Access Monitoring \u2014 Real-time logging across compute/data/control planes",
        "Anomaly Detection \u2014 4 automatic detectors (unusual time, excess data, escalation, PII export)",
        "Business Glossary \u2014 Self-contained (no Purview required) with policy engine",
        "Compliance Reports \u2014 SOC 2, HIPAA, audit trail dashboards"
    ],
    subtitle="Enterprise-grade security built in"
)

# === Slide 9: Data Integration ===
add_content_slide(
    prs,
    "Data Integration",
    [
        "Fabric Mirroring \u2014 Real-time CDC from SQL, Oracle, Cosmos DB (< 30s latency)",
        "SFTP Exchange \u2014 Secure data sharing with TPAs, reinsurers, regulators",
        "External Datasets \u2014 15+ free datasets (CDC, Census, FRED, NOAA)",
        "Streaming \u2014 EventHub connectors for real-time claims/payments",
        "API Integration \u2014 REST API for bulk import/export",
        "PGP Encryption \u2014 HIPAA/PCI-DSS compliant data exchange"
    ],
    subtitle="Connect to any data source"
)

# === Slide 10: Deployment ===
add_content_slide(
    prs,
    "Deployment & DevOps",
    [
        "Infrastructure as Code \u2014 Terraform, Bicep, Fabric CLI templates",
        "Multi-Environment \u2014 Dev (F2), Test (F4), Production (F64)",
        "CI/CD Automation \u2014 GitHub Actions + Azure DevOps pipelines",
        "Automated Testing \u2014 Unit, integration, security, DQ tests",
        "Marketplace Ready \u2014 Installation wizard + auto-configuration",
        "Git Integration \u2014 Bi-directional sync with GitHub/Azure DevOps"
    ],
    subtitle="Production-ready from day one"
)

# === Slide 11: Quick Start ===
add_content_slide(
    prs,
    "Quick Start (30 Minutes)",
    [
        "Step 1: Import notebooks to Fabric workspace (2 min)",
        "Step 2: Run Notebook 01 \u2014 generates demo data (5 min)",
        "Step 3: Run Notebook 30 \u2014 medallion transformations (10 min)",
        "Step 4: Run Notebook 90 \u2014 view dashboard (2 min)",
        "Step 5: Explore AI features via Notebook 20 (10 min)",
        "Done! Full solution running with demo data"
    ],
    subtitle="Demo to production in under 30 minutes"
)

# === Slide 12: Contact / Thank You ===
add_title_slide(
    prs,
    "Thank You",
    "Insurance Data & AI Accelerator \u2022 Microsoft Fabric"
)

# Save
master_ppt_path = f"{output_dir}/PPT_00_Accelerator_Master_Presentation.pptx"
prs.save(master_ppt_path)
print(f"\n\u2705 Master Accelerator PPT saved: {master_ppt_path}")
print("=" * 80)

In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5: Runbook Generator                                       ║
# ╚════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("📝 GENERATING RUNBOOKS")
print("=" * 80)

runbook_dir = "docs/runbooks"
os.makedirs(runbook_dir, exist_ok=True)

for feature in FEATURE_REGISTRY:
    runbook_content = f"""# \u2699\ufe0f Runbook: {feature['title']}

> **Notebook**: `{feature['notebook']}.ipynb`  
> **Category**: {feature['category']}  
> **Last Updated**: {datetime.now().strftime('%Y-%m-%d')}  
> **Owner**: Platform Engineering  

---

## 1. Overview

{feature['summary']}

**Target Personas**: {', '.join(feature['personas'])}

## 2. Prerequisites

### Required Access
- Fabric workspace contributor access
- Lakehouse read/write permissions

### Dependencies
{chr(10).join(f'- {dep}' for dep in feature['dependencies']) if feature['dependencies'] else '- None'}

### Required Lakehouses
- `insurance_bronze` (raw data)
- `insurance_silver` (cleansed)
- `insurance_gold` (business-ready)
- `insurance_metadata` (configuration)

## 3. Execution Steps

### Step 1: Open Notebook
1. Navigate to Fabric workspace
2. Open `{feature['notebook']}.ipynb`
3. Verify attached lakehouse is correct

### Step 2: Configure Parameters
1. Review Section 1 configuration cell
2. Update environment-specific settings if needed
3. Verify Key Vault references (if applicable)

### Step 3: Execute
1. Click \"Run All\" to execute all cells sequentially
2. Monitor progress in cell outputs
3. Expected duration: {feature['sla']}

### Step 4: Verify Results
1. Check output tables exist
2. Verify record counts match expectations
3. Review any warnings or alerts

## 4. Expected Outputs

| Output | Description |
|--------|-------------|
{chr(10).join(f'| `{out}` | Generated by this notebook |' for out in feature['outputs'])}

## 5. Monitoring & Alerting

### Health Check
```python
# Verify notebook ran successfully
result = spark.sql(\"\"\"\n    SELECT * FROM metadata.pipeline_execution_log\n    WHERE notebook_name = '{feature['notebook']}'\n    ORDER BY start_timestamp DESC\n    LIMIT 1\n\"\"\")
display(result)
```

### SLA Monitoring
- **Target SLA**: {feature['sla']}
- **Alert threshold**: 2x SLA target
- **Escalation**: Platform Engineering team

## 6. Troubleshooting

### Common Issues

| Issue | Cause | Resolution |
|-------|-------|------------|
| Notebook fails on cell 1 | Missing utility notebook | Ensure `00_fabric_native_utils` exists |
| Lakehouse not found | Wrong attachment | Re-attach correct lakehouse |
| Timeout | Large data volume | Increase Spark executor memory |
| Permission denied | RBAC issue | Verify workspace role assignment |

### Recovery Procedures

1. **Partial failure**: Re-run from failed cell (idempotent design)
2. **Data corruption**: Use Delta Time Travel to rollback
   ```sql
   RESTORE TABLE target_table TO VERSION AS OF <version_number>
   ```
3. **Full re-run**: Safe to re-execute entire notebook

## 7. Maintenance

### Scheduled Execution
- **Frequency**: Based on business requirements
- **Tool**: Fabric Data Pipeline
- **Monitoring**: Notebook 45 (Operational Monitoring)

### Version History
| Version | Date | Change |
|---------|------|--------|
| 1.0 | {datetime.now().strftime('%Y-%m-%d')} | Initial release |

---
*Generated by Documentation Generator (Notebook 75)*
"""
    
    # Save runbook
    filename = f"{runbook_dir}/RUNBOOK_{feature['id']}_{feature['notebook']}.md"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(runbook_content)
    
    print(f"   \u2705 {feature['title']} \u2192 {filename}")

print(f"\n\u2705 Generated {len(FEATURE_REGISTRY)} runbooks")
print("=" * 80)

In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  SECTION 6: Design Document Generator                               ║
# ╚════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("📝 GENERATING DESIGN DOCUMENTS")
print("=" * 80)

design_dir = "docs/design_documents"
os.makedirs(design_dir, exist_ok=True)

for feature in FEATURE_REGISTRY:
    design_doc = f"""# \ud83d\udcd0 Design Document: {feature['title']}

> **Document ID**: DD-{feature['id']}  
> **Version**: 1.0  
> **Status**: Approved  
> **Date**: {datetime.now().strftime('%Y-%m-%d')}  
> **Author**: Architecture Team  

---

## 1. Introduction

### 1.1 Purpose
This document describes the technical design of **{feature['title']}** within the Insurance Fabric Accelerator.

### 1.2 Scope
{feature['summary']}

### 1.3 Target Audience
{', '.join(feature['personas'])}

## 2. Architecture

### 2.1 High-Level Design
{feature['architecture']}

### 2.2 Component Diagram
```
\u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510    \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510    \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510
\u2502     INPUTS        \u2502 \u2192  \u2502   PROCESSING     \u2502 \u2192  \u2502     OUTPUTS       \u2502
\u2502                   \u2502    \u2502                   \u2502    \u2502                   \u2502
\u2502 {', '.join(feature['inputs'])[:17].ljust(17)} \u2502    \u2502 {feature['notebook'][:17].ljust(17)} \u2502    \u2502 {', '.join(feature['outputs'])[:17].ljust(17)} \u2502
\u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518    \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518    \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518
```

### 2.3 Data Flow

**Inputs:**
{chr(10).join(f'- `{inp}`' for inp in feature['inputs'])}

**Outputs:**
{chr(10).join(f'- `{out}`' for out in feature['outputs'])}

## 3. Technical Specifications

### 3.1 Key Capabilities
{chr(10).join(f'{i+1}. **{cap}**' for i, cap in enumerate(feature['key_capabilities']))}

### 3.2 Dependencies
| Dependency | Type | Required |
|-----------|------|----------|
{chr(10).join(f'| {dep} | Notebook | Yes |' for dep in feature['dependencies']) if feature['dependencies'] else '| None | - | - |'}

### 3.3 Configuration
- Environment: Fabric workspace with attached lakehouses
- Compute: Spark cluster (auto-scaling recommended)
- Storage: Delta Lake format for all tables

## 4. Performance

### 4.1 SLA Targets
| Metric | Target |
|--------|--------|
| Execution Time | {feature['sla']} |
| Data Freshness | Based on update frequency |

### 4.2 Scalability
- Spark auto-scaling handles data volume growth
- Partitioned Delta tables for query performance
- Z-Order optimization for frequent access patterns

## 5. Security

### 5.1 Authentication
- Entra ID (Azure AD) for all user access
- Managed Identity for service-to-service
- Key Vault for secrets management

### 5.2 Authorization
- Workspace RBAC (Admin, Contributor, Viewer)
- Row-Level Security on semantic models
- Column masking for PII/PCI fields

## 6. Testing Strategy

### 6.1 Test Types
| Test Type | Coverage | Notebook |
|-----------|----------|----------|
| Unit Tests | Core functions | Notebook 76 |
| Integration Tests | End-to-end flow | Notebook 76 |
| Performance Tests | SLA validation | Notebook 76 |

### 6.2 Demo Data
{feature['demo_data']}

## 7. Deployment

### 7.1 Environments
| Environment | Capacity | Purpose |
|-------------|----------|----------|
| Dev | F2 | Development and testing |
| Test | F4 | Integration testing |
| Prod | F64 | Production workloads |

### 7.2 Rollback
- Delta Lake Time Travel for data rollback
- Git-based notebook versioning
- Idempotent execution (safe to re-run)

## 8. Appendix

### 8.1 Related Documents
- Runbook: `RUNBOOK_{feature['id']}_{feature['notebook']}.md`
- Presentation: `PPT_{feature['id']}_{feature['notebook']}.pptx`
- Test Cases: `TESTCASES_{feature['id']}_{feature['notebook']}.md`

### 8.2 Change Log
| Version | Date | Author | Change |
|---------|------|--------|--------|
| 1.0 | {datetime.now().strftime('%Y-%m-%d')} | Architecture Team | Initial release |

---
*Generated by Documentation Generator (Notebook 75)*
"""
    
    # Save design doc
    filename = f"{design_dir}/DESIGN_{feature['id']}_{feature['notebook']}.md"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(design_doc)
    
    print(f"   \u2705 {feature['title']} \u2192 {filename}")

print(f"\n\u2705 Generated {len(FEATURE_REGISTRY)} design documents")
print("=" * 80)

In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  SECTION 7: Summary Report                                          ║
# ╚════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("📊 DOCUMENTATION GENERATION SUMMARY")
print("=" * 80)

total_features = len(FEATURE_REGISTRY)

print(f"\n   Features Documented: {total_features}")
print(f"   \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
print(f"   \ud83d\udcca PowerPoint Presentations: {total_features + 1}")
print(f"      \u251c\u2500 Per-Feature PPTs: {total_features}")
print(f"      \u2514\u2500 Master Accelerator PPT: 1")
print(f"   \ud83d\udcdd Runbooks: {total_features}")
print(f"   \ud83d\udcd0 Design Documents: {total_features}")
print(f"   \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
print(f"   Total Documents: {total_features * 3 + 1}")
print(f"")
print(f"   \ud83d\udcc1 Output Directories:")
print(f"      \u251c\u2500 docs/presentations/   ({total_features + 1} .pptx files)")
print(f"      \u251c\u2500 docs/runbooks/        ({total_features} .md files)")
print(f"      \u2514\u2500 docs/design_documents/ ({total_features} .md files)")
print(f"")
print("=" * 80)
print("\n\u2705 All documentation generated successfully!")